In [ ]:
# IMPORTS
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score
)

import torchxrayvision as xrv
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Reproducibility
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
torch.cuda.manual_seed_all(GLOBAL_SEED)


In [ ]:
class Config:
    pathology = "Pneumothorax"  # edit to run for a different pathology
    base_dir = "/path/to/dataset_root"     # placeholder — edit to your local image root
    csv_path = "/path/to/pneumothorax.csv"  # placeholder — edit to your local labels CSV
    result_root = "results_final"

    # Choose whether to train or only run inference
    train_new_model = True       # Set to False to load pretrained weights manually

    # Path to model to load if train_new_model=False (placeholder — edit as needed)
    pretrained_model_path = "/path/to/results_final/experiment_pneumothorax/best_Pneumothorax.pth"

    # Training parameters
    epochs = 16
    patience = 2
    batch_size = 32
    lr = 1e-4
    weight_decay = 1e-4
    random_state = 42

cfg = Config()
print(cfg.pathology)
print(cfg.pretrained_model_path)


In [ ]:
def get_device():
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
    print(f"Using device: {device}")
    return device

device = get_device()


In [ ]:
def make_result_dir(pathology_name, base_root="results", prefix=None):
    """Create timestamped result directory."""
    timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    parts = [p for p in [prefix, pathology_name.lower(), timestamp] if p]
    path = os.path.join(base_root, "_".join(parts))
    os.makedirs(path, exist_ok=True)
    print(f"Results directory: {path}")
    return path


In [ ]:
class CXRBinaryDataset(Dataset):
    """Binary classification dataset with deterministic path reconstruction."""

    def __init__(self, dataframe, base_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.base_dir = base_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = str(row["filename"])
        subject_id = str(row["subject_id"])
        study_id = str(row["study_id"])
        label = torch.tensor(row["label"], dtype=torch.float32)

        # Build the full image path
        patient_prefix = subject_id[:2]
        img_path = os.path.join(
            self.base_dir,
            f"p{patient_prefix}",
            f"p{subject_id}",
            f"s{study_id}",
            filename
        )

        if not os.path.exists(img_path):
            img = Image.new("RGB", (224, 224), color=(0, 0, 0))
        else:
            img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        return img, label, row.to_dict()


In [ ]:
# TRANSFORMS
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])


In [ ]:
def create_splits(
    csv_path,
    pathology_col="Pneumothorax",
    val_size=0.15,
    test_size=0.15,
    random_state=42,
    result_dir=None,
    balance_classes=True,
    fraction=1.0,
    split_dir=None,  # optional: folder to look for existing split CSVs (defaults to result_dir)
):
    """
    Create or load patient-disjoint splits for binary classification.
    If CSV splits already exist in `split_dir`, they will be loaded instead of regenerated.

    Args:
        csv_path: Path to full dataset CSV.
        pathology_col: Pathology name (column to use as positive label).
        val_size, test_size: Fraction of positives for validation/test.
        random_state: Random seed.
        result_dir: Directory where new splits will be saved (if created).
        balance_classes: Whether to balance classes 1:1 inside each split.
        fraction: Fraction of the dataset to use (0 < fraction <= 1.0).
        split_dir: Optional custom folder to look for existing split CSVs.
    """
    import numpy as np
    import pandas as pd
    from sklearn.model_selection import train_test_split
    import os

    # 1 Check for existing splits
    if split_dir is None:
        split_dir = result_dir

    expected_files = {
        "train": os.path.join(split_dir, f"split_{pathology_col.lower()}_train.csv"),
        "val":   os.path.join(split_dir, f"split_{pathology_col.lower()}_val.csv"),
        "test":  os.path.join(split_dir, f"split_{pathology_col.lower()}_test.csv"),
    }

    if all(os.path.exists(p) for p in expected_files.values()):
        print(f"Found existing splits in: {split_dir}")
        train_df = pd.read_csv(expected_files["train"])
        val_df = pd.read_csv(expected_files["val"])
        test_df = pd.read_csv(expected_files["test"])
        print("Loaded existing split CSVs.")
        return train_df, val_df, test_df

    print(f"No existing splits found creating new ones under: {result_dir}")

    # 2 Build dataset
    df = pd.read_csv(csv_path)
    df["subject_id"] = df["subject_id"].astype(str)

    no_pathology_col = f"No {pathology_col}"
    if no_pathology_col in df.columns:
        df = df[~((df[pathology_col] == 1) & (df[no_pathology_col] == 1))]

    pos_df = df[df[pathology_col] == 1].copy()
    neg_df = (
        df[df[no_pathology_col] == 1].copy()
        if no_pathology_col in df.columns
        else df[df[pathology_col] == 0].copy()
    )

    print(f"Found {len(pos_df)} positive and {len(neg_df)} negative samples before balancing.")

    # 3 Optional subsampling
    if fraction < 1.0:
        n_pos = max(1, int(len(pos_df) * fraction))
        n_neg = max(1, int(len(neg_df) * fraction))
        pos_df = pos_df.sample(n=n_pos, random_state=random_state)
        neg_df = neg_df.sample(n=n_neg, random_state=random_state)
        print(f"Using only {fraction*100:.1f}% of dataset {n_pos} pos / {n_neg} neg.")

    # Assign labels
    pos_df["label"], neg_df["label"] = 1, 0

    # 4 Patient-disjoint split
    patients = pd.concat([pos_df, neg_df])["subject_id"].unique()
    train_p, temp_p = train_test_split(
        patients, test_size=(val_size + test_size), random_state=random_state
    )
    val_p, test_p = train_test_split(
        temp_p, test_size=test_size / (val_size + test_size), random_state=random_state
    )

    def subset(df, ids): return df[df["subject_id"].isin(ids)]
    train_df = pd.concat([subset(pos_df, train_p), subset(neg_df, train_p)])
    val_df = pd.concat([subset(pos_df, val_p), subset(neg_df, val_p)])
    test_df = pd.concat([subset(pos_df, test_p), subset(neg_df, test_p)])

    # 5 Balance classes inside each split
    def balance_split(df_split):
        if not balance_classes:
            return df_split
        n_pos = (df_split["label"] == 1).sum()
        n_neg = (df_split["label"] == 0).sum()
        n_min = min(n_pos, n_neg)
        pos_part = df_split[df_split["label"] == 1].sample(n=n_min, random_state=random_state)
        neg_part = df_split[df_split["label"] == 0].sample(n=n_min, random_state=random_state)
        return pd.concat([pos_part, neg_part]).sample(frac=1, random_state=random_state).reset_index(drop=True)

    train_df = balance_split(train_df)
    val_df = balance_split(val_df)
    test_df = balance_split(test_df)

    print(f"Final balanced sizes: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

    # 6 Save new splits
    if result_dir:
        os.makedirs(result_dir, exist_ok=True)
        for name, dset in [("train", train_df), ("val", val_df), ("test", test_df)]:
            out_path = expected_files[name]
            dset.to_csv(out_path, index=False)
        print(f"Saved splits to {result_dir}")

    return train_df, val_df, test_df


In [ ]:
def save_split_summary(train_df, val_df, test_df, pathology_name, result_dir):
    """
    Save a CSV summarizing how many positive and negative samples are present in each split.
    """
    os.makedirs(result_dir, exist_ok=True)

    def count_labels(df):
        n_total = len(df)
        n_pos = int((df["label"] == 1).sum())
        n_neg = int((df["label"] == 0).sum())
        return n_total, n_pos, n_neg, n_pos / max(n_total, 1)

    summary = []
    for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
        n_total, n_pos, n_neg, frac_pos = count_labels(df)
        summary.append({
            "Split": name,
            "Total": n_total,
            "Positives": n_pos,
            "Negatives": n_neg,
            "Positive_Fraction": round(frac_pos, 3)
        })

    summary_df = pd.DataFrame(summary)
    summary_csv = os.path.join(result_dir, f"split_summary_{pathology_name}.csv")
    summary_df.to_csv(summary_csv, index=False, sep=";")
    print(f"Saved split summary {summary_csv}")
    display(summary_df)  # shows nicely inside Jupyter


In [ ]:
def check_patient_overlap(train_df, val_df, test_df):
    t1, t2, t3 = set(train_df.subject_id), set(val_df.subject_id), set(test_df.subject_id)
    overlap = (t1 & t2) | (t1 & t3) | (t2 & t3)
    if overlap:
        print(f"Warning: {len(overlap)} patients appear in multiple splits!")
    else:
        print("Patient-disjoint verified: no patient overlap between splits.")



In [ ]:
# MODEL DEFINITION
def build_xrv_model(num_classes: int, label_cols, unfreeze_last_blocks: int = 1):
    model = xrv.models.DenseNet(weights="densenet121-res224-chex")
    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, num_classes)
    model.op_threshs = None
    model.n_outputs = num_classes
    model.classes = list(label_cols)

    for p in model.parameters(): p.requires_grad = False
    for p in model.classifier.parameters(): p.requires_grad = True

    if unfreeze_last_blocks > 0:
        blocks = list(model.features.named_children())
        for _, module in blocks[-unfreeze_last_blocks:]:
            for p in module.parameters():
                p.requires_grad = True
    return model


In [ ]:
# TRAINING LOOP
def train_model(model, train_loader, val_loader, device, result_dir, pathology_name, epochs=50, patience=4, lr=1e-4, wd=1e-4):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=wd)
    best_val_loss = float("inf")
    no_improve = 0

    train_losses, val_losses = [], []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0
        for xb, yb, _ in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", ncols=100):
            xb, yb = xb.to(device), yb.to(device).unsqueeze(1)
            xb = xb.mean(1, keepdim=True)
            mu, sd = xb.mean((2,3), keepdim=True), xb.std((2,3), keepdim=True) + 1e-8
            xb = (xb - mu) / sd
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)

        train_loss = running_loss / len(train_loader.dataset)
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for xb, yb, _ in val_loader:
                xb, yb = xb.to(device), yb.to(device).unsqueeze(1)
                xb = xb.mean(1, keepdim=True)
                mu, sd = xb.mean((2,3), keepdim=True), xb.std((2,3), keepdim=True) + 1e-8
                xb = (xb - mu) / sd
                val_loss += criterion(model(xb), yb).item() * xb.size(0)
        val_loss /= len(val_loader.dataset)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(f"Epoch {epoch}: Train Loss={train_loss:.4f} | Val Loss={val_loss:.4f}")
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), os.path.join(result_dir, f"best_{pathology_name}.pth"))
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print("Early stopping.")
                break

    # Save loss curve
    plt.figure()
    plt.plot(range(1, len(train_losses)+1), train_losses, label="Train")
    plt.plot(range(1, len(val_losses)+1), val_losses, label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.title(f"{pathology_name} – Training Curve")
    path_plot = os.path.join(result_dir, f"loss_curve_{pathology_name}.png")
    plt.savefig(path_plot, dpi=150)
    plt.close()
    print(f"Saved training curve {path_plot}")


In [ ]:
@torch.no_grad()
def evaluate_model(model, dataloader, dataset_df, device, pathology_name, result_dir):
    """
    Evaluate model on a test set with multiple thresholds:
    - Fixed 0.5
    - Youden's J statistic
    - F1-optimized threshold

    Saves ROC, confusion matrices, and a CSV with predictions.
    """
    os.makedirs(result_dir, exist_ok=True)
    model.eval()

    y_true, y_prob = [], []
    for xb, yb, _ in tqdm(dataloader, desc="Inference", ncols=100):
        xb, yb = xb.to(device), yb.to(device).unsqueeze(1)
        xb = xb.mean(1, keepdim=True)
        mu, sd = xb.mean((2,3), keepdim=True), xb.std((2,3), keepdim=True) + 1e-8
        xb = (xb - mu) / sd
        logits = model(xb)
        y_true.extend(yb.cpu().numpy().flatten())
        y_prob.extend(torch.sigmoid(logits).cpu().numpy().flatten())

    y_true = np.array(y_true)
    y_prob = np.clip(np.array(y_prob), 0, 1)

    # ROC and optimal thresholds
    fpr, tpr, thr_roc = roc_curve(y_true, y_prob)
    j_stat = tpr - fpr
    thr_youden = thr_roc[np.argmax(j_stat)]

    prec, rec, thr_pr = precision_recall_curve(y_true, y_prob)
    f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
    thr_f1 = thr_pr[np.argmax(f1)]

    auc_roc = roc_auc_score(y_true, y_prob)
    auc_pr = average_precision_score(y_true, y_prob)

    print(f"\nThreshold optimization for {pathology_name}:")
    print(f" - Fixed 0.5")
    print(f" - Youden J @ {thr_youden:.3f}")
    print(f" - F1 @ {thr_f1:.3f}")
    print(f"AUC ROC={auc_roc:.4f} | AUC PR={auc_pr:.4f}")

    thresholds = {"fixed_0.5": 0.5, "youden": thr_youden, "f1": thr_f1}

    # Save ROC curve
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, lw=2, color="darkorange", label=f"AUC = {auc_roc:.3f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve – {pathology_name}")
    plt.legend()
    plt.tight_layout()
    roc_path = os.path.join(result_dir, f"roc_{pathology_name}.png")
    plt.savefig(roc_path, dpi=200)
    plt.close()
    print(f"Saved ROC curve {roc_path}")

    # Evaluate and save confusion matrices
    metrics_summary = []
    for name, thr in thresholds.items():
        y_pred = (y_prob >= thr).astype(int)
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        acc = (tp + tn) / (tn + fp + fn + tp)
        prec_val = tp / max(tp + fp, 1)
        rec_val = tp / max(tp + fn, 1)
        f1_val = 2 * prec_val * rec_val / max(prec_val + rec_val, 1e-12)
        specificity = tn / max(tn + fp, 1)
        npv = tn / max(tn + fn, 1)

        metrics_summary.append({
            "Threshold Type": name,
            "Threshold Value": thr,
            "TP": tp, "FP": fp, "TN": tn, "FN": fn,
            "Accuracy": acc, "Precision": prec_val,
            "Recall": rec_val, "Specificity": specificity,
            "NPV": npv, "F1": f1_val
        })

        # Confusion matrix plot
        plt.figure(figsize=(4, 4))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                    xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
        plt.title(f"{pathology_name} – {name.upper()} (thr={thr:.2f})")
        plt.tight_layout()
        cm_path = os.path.join(result_dir, f"cm_{pathology_name}_{name}.png")
        plt.savefig(cm_path, dpi=200)
        plt.close()
        print(f"Saved confusion matrix ({name}) {cm_path}")

    # Save prediction CSV
    df_out = dataset_df.copy()
    df_out["true_label"] = y_true
    df_out["pred_prob"] = y_prob
    df_out["pred_label_thr0.5"] = (y_prob >= 0.5).astype(int)
    df_out["pred_label_thr_youden"] = (y_prob >= thr_youden).astype(int)
    df_out["pred_label_thr_f1"] = (y_prob >= thr_f1).astype(int)

    pred_csv = os.path.join(result_dir, f"predictions_{pathology_name}.csv")
    df_out.to_csv(pred_csv, index=False, sep=";")
    print(f"Saved predictions CSV {pred_csv}")

    # Save metrics summary
    metrics_df = pd.DataFrame(metrics_summary)
    metrics_csv = os.path.join(result_dir, f"metrics_summary_{pathology_name}.csv")
    metrics_df.to_csv(metrics_csv, index=False, sep=";")
    print(f"Saved metrics summary {metrics_csv}")

    return {
        "auc_roc": auc_roc,
        "auc_pr": auc_pr,
        "roc_curve": roc_path,
        "predictions_csv": pred_csv,
        "metrics_csv": metrics_csv
    }


In [ ]:
# MAIN EXECUTION
result_dir = make_result_dir(cfg.pathology, cfg.result_root, prefix="experiment")

# Use existing splits if they exist, otherwise create new ones
train_df, val_df, test_df = create_splits(
    csv_path=cfg.csv_path,
    pathology_col=cfg.pathology,
    result_dir=result_dir,
    split_dir=cfg.split_dir if hasattr(cfg, "split_dir") else None,  # optional
    balance_classes=True,
    fraction=1.0,
)


save_split_summary(train_df, val_df, test_df, cfg.pathology, result_dir)
check_patient_overlap(train_df, val_df, test_df)


train_dataset = CXRBinaryDataset(train_df, cfg.base_dir, transform=train_transform)
val_dataset = CXRBinaryDataset(val_df, cfg.base_dir, transform=eval_transform)
test_dataset = CXRBinaryDataset(test_df, cfg.base_dir, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

# Build model
model = build_xrv_model(num_classes=1, label_cols=[cfg.pathology]).to(device)
best_path = os.path.join(result_dir, f"best_{cfg.pathology}.pth")

# Training or Loading
if cfg.train_new_model:
    print("Starting fine-tuning...")
    train_model(model, train_loader, val_loader, device, result_dir, cfg.pathology,
                epochs=cfg.epochs, patience=cfg.patience, lr=cfg.lr, wd=cfg.weight_decay)
else:
    # Use user-defined pretrained model path
    path_model = cfg.pretrained_model_path
    if not os.path.exists(path_model):
        raise FileNotFoundError(f"Pretrained model not found at: {path_model}")
    print(f"Loading pretrained weights from: {path_model}")
    model.load_state_dict(torch.load(path_model, map_location=device))

# Evaluation
print("\nEvaluating model on test set...")
results = evaluate_model(model, test_loader, test_df, device, cfg.pathology, result_dir)

print("\n Pipeline complete!")
print(f"AUC ROC: {results['auc_roc']:.4f}")
print(f"AUC PR:  {results['auc_pr']:.4f}")
print(f"Predictions CSV: {results['predictions_csv']}")
print(f"Metrics CSV: {results['metrics_csv']}")
